# Experiment: AHSP Buffer Gate

Objective:
- Decide whether alpha-hemoglobin-stabilizing protein (`AHSP`) should become a therapy candidate, an assay readout, or a rejected lane.
- Use only repository snapshots from 2026-04-27 and keep the decision reproducible.

Success criteria:
- Preserve AHSP if it improves the dual HbF plus alpha-globin assay logic.
- Reject standalone-therapy framing if the evidence says AHSP alone is not enough.


In [ ]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path.cwd()
PUBMED_DIR = ROOT / "data/literature/pubmed"
SNAPSHOT_PATHS = {
    "broad_ahsp": PUBMED_DIR
    / ("2026-04-27-ahsp-beta-thalassemia-alpha-globin-search.json"),
    "overexpression": PUBMED_DIR
    / ("2026-04-27-ahsp-overexpression-beta-thalassemia-search.json"),
    "nrf2_ahsp": PUBMED_DIR / "2026-04-27-nrf2-ahsp-beta-thalassemia-search.json",
    "sirolimus_ahsp": PUBMED_DIR
    / "2026-04-27-sirolimus-ahsp-beta-thalassemia-search.json",
}
ABSTRACT_PATH = PUBMED_DIR / "2026-04-27-ahsp-selected-abstracts.xml"

## Plan

Hypotheses:
- `AHSP` is useful as an alpha-globin-buffer readout because beta-thalassemia injury includes excess free alpha-globin.
- `AHSP` should not become a standalone therapy lead unless intervention evidence shows hematologic rescue.

Metrics:
- Direct beta-thalassemia relevance.
- Direct alpha-globin-buffer relevance.
- Direct bridge to existing Nakafa lanes: `NRF2`, sirolimus, alpha-globin cleanup.
- Standalone intervention caution.


In [ ]:
def load_snapshot(path: Path) -> dict:
    """Load one compact PubMed search snapshot."""
    with path.open(encoding="utf-8") as snapshot_file:
        return json.load(snapshot_file)


def collect_records(snapshot_paths: dict[str, Path]) -> dict[str, dict]:
    """Collect and de-duplicate PubMed records across snapshot files."""
    records: dict[str, dict] = {}

    for source_name, path in snapshot_paths.items():
        snapshot = load_snapshot(path)
        for record in snapshot.get("results", []):
            pmid = str(record.get("pmid", ""))
            if not pmid:
                continue

            existing = records.setdefault(
                pmid,
                {
                    "pmid": pmid,
                    "title": record.get("title", ""),
                    "journal": record.get("journal", ""),
                    "pubdate": record.get("pubdate", ""),
                    "sources": [],
                },
            )
            existing["sources"].append(source_name)

    return records


def parse_abstract_titles(path: Path) -> dict[str, str]:
    """Return PMID-to-title mapping from a saved PubMed XML efetch file."""
    root = ET.parse(path).getroot()
    titles = {}

    for article in root.findall("PubmedArticle"):
        pmid = article.findtext("./MedlineCitation/PMID")
        title_node = article.find("./MedlineCitation/Article/ArticleTitle")
        if pmid and title_node is not None:
            titles[pmid] = "".join(title_node.itertext())

    return titles


records = collect_records(SNAPSHOT_PATHS)
selected_titles = parse_abstract_titles(ABSTRACT_PATH)

summary = {
    "unique_pubmed_records": len(records),
    "selected_abstracts": len(selected_titles),
    "snapshot_counts": {
        name: load_snapshot(path).get("count", 0)
        for name, path in SNAPSHOT_PATHS.items()
    },
}
summary

In [ ]:
def role_labels(record: dict) -> list[str]:
    """Assign compact evidence roles from title text and snapshot source."""
    title = str(record.get("title", "")).lower()
    sources = set(record.get("sources", []))
    labels = []

    if "nrf2" in title or "nrf2_ahsp" in sources:
        labels.append("nrf2_to_ahsp_bridge")
    if "sirolimus" in title or "sirolimus_ahsp" in sources:
        labels.append("sirolimus_readout_support")
    if "overexpression" in title or "overexpression" in sources:
        labels.append("standalone_overexpression_caution")
    if "modulatory" in title or "modifier" in title or "molecular chaperone" in title:
        labels.append("alpha_globin_buffer_biology")

    return labels


evidence_rows = []
for pmid, record in sorted(records.items()):
    labels = role_labels(record)
    if not labels:
        continue

    evidence_rows.append(
        {
            "pmid": pmid,
            "labels": labels,
            "title": record["title"],
            "sources": sorted(record["sources"]),
        }
    )

evidence_rows

In [ ]:
gate_checks = [
    {
        "gate": "alpha_globin_buffer_biology",
        "status": "positive",
        "weight": 2,
        "pmids": ["21490703", "31894534"],
        "meaning": "AHSP is biologically tied to free alpha-globin handling.",
    },
    {
        "gate": "nrf2_to_ahsp_bridge",
        "status": "positive",
        "weight": 2,
        "pmids": ["35092867"],
        "meaning": "NRF2 can increase AHSP in beta-thalassemia model context.",
    },
    {
        "gate": "sirolimus_readout_support",
        "status": "positive",
        "weight": 2,
        "pmids": ["38731008"],
        "meaning": (
            "Sirolimus-treated beta-thalassemia ErPCs can show AHSP mRNA increase."
        ),
    },
    {
        "gate": "standalone_overexpression_caution",
        "status": "caution",
        "weight": -3,
        "pmids": ["20815047"],
        "meaning": (
            "Supraphysiologic AHSP overexpression did not materially "
            "rescue murine beta-thalassemia intermedia."
        ),
    },
]

readout_score = sum(
    check["weight"] for check in gate_checks if check["status"] == "positive"
)
therapy_score = sum(
    check["weight"] for check in gate_checks if check["status"] != "positive"
)

decision = {
    "lane": "AHSP_alpha_globin_buffer",
    "readout_score": readout_score,
    "standalone_therapy_score": therapy_score,
    "decision": "add_AHSP_as_expanded_readout_not_lead",
    "required_pairing": [
        "HbF protein or F-cell percentage",
        "alpha/non-alpha globin balance",
        "free or insoluble alpha-globin",
        "ULK1 and p62/SQSTM1 autophagy context",
        "erythroid maturation",
        "viability and hemolysis",
    ],
}

assert readout_score > 0
assert therapy_score < 0
assert decision["decision"] == "add_AHSP_as_expanded_readout_not_lead"

decision

## Results

Key observations:
- AHSP has direct disease-mechanism relevance because it binds/free-buffers alpha-globin biology.
- The NRF2 result makes AHSP a better interpretation bridge for the existing `AMPKB1` / `NRF2` expansion lane.
- The sirolimus result makes AHSP a useful optional endpoint beside `ULK1`, p62/`SQSTM1`, and alpha-globin burden.
- The mouse overexpression result blocks simplistic `more AHSP = cure` framing.

Decision:
- Add AHSP as an expanded dual-readout endpoint where a lab can measure it.
- Do not promote AHSP induction or overexpression as a standalone therapy lane.


## Next Steps

- Move the durable conclusion into a finding.
- Update the dual-readout assay spec and lab quote brief with AHSP as optional/expanded, not mandatory for the first quote.
- Update the source registry and paper so AHSP is traceable as a buffer/readout gate.
